In [1]:
import hmac
import time
import numpy as np
import hashlib
import os

from cryptography.hazmat.primitives.asymmetric import x25519, ed25519
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import serialization

# User Registration

Initialization of users Alice and Bob

the registration phase consists of a party generating a key bundle consisting of:

1. one identity key (long term)
2. one signed pre-key (mid term)
3. multiple ephemeral keys (short term)

Key signature is done using the identity key

In [2]:
class KeyBundle:
    def __init__(self):
        # Identity Key (Long-term)
        self.identity_key = x25519.X25519PrivateKey.generate()
        self.signing_key = ed25519.Ed25519PrivateKey.generate()
        
        # Signed Pre-key (Mid-term)
        self.signed_pre_key = x25519.X25519PrivateKey.generate()
        
        # Signature of the signed pre-key public bytes using the identity signing key
        public_pre_key_bytes = self.signed_pre_key.public_key().public_bytes(
            encoding=serialization.Encoding.Raw,
            format=serialization.PublicFormat.Raw
        )
        self.signed_pre_key_signature = self.signing_key.sign(public_pre_key_bytes)
        
        # One-time Ephemeral Keys (Short-term)
        self.ephemeral_keys = [x25519.X25519PrivateKey.generate() for _ in range(10)]


    def get_public_keys(self):
        return {
            'ik': self.identity_key.public_key(),
            'spk': self.signed_pre_key.public_key(),
            'spk_sig': self.signed_pre_key_signature,
            'otks': [k.public_key() for k in self.ephemeral_keys],
            'sig_pk': self.signing_key.public_key()
        }
        
    def verify_spk(self, bundle):
        spk_bytes = bundle['spk'].public_bytes(
            encoding=serialization.Encoding.Raw,
            format=serialization.PublicFormat.Raw
        )
        bundle['sig_pk'].verify(bundle['spk_sig'], spk_bytes)

In [3]:
start_time = time.time()
alice = KeyBundle()
bob = KeyBundle()
registration_time = time.time() - start_time

# Alice receives Bob's public key bundle
bob_bundle = bob.get_public_keys()

# Alice verifies Bob's SPK signature
alice.verify_spk(bob_bundle)

print(f"Registration Time (Alice + Bob): {registration_time:.6f} seconds")
print("Bob's Signed Pre-key verified successfully by Alice.")

Registration Time (Alice + Bob): 0.010555 seconds


# Extended Triple Diffie Hellman

Alice computes her master secret

In [4]:
start_time = time.time()
# Alice uses Bob's bundle to compute her master secret
alice_dh_1 = alice.identity_key.exchange(bob_bundle['spk'])
alice_dh_2 = alice.ephemeral_keys[0].exchange(bob_bundle['ik'])
alice_dh_3 = alice.ephemeral_keys[0].exchange(bob_bundle['spk'])
alice_dh_4 = alice.ephemeral_keys[0].exchange(bob_bundle['otks'][0])

alice_ms = alice_dh_1 + alice_dh_2 + alice_dh_3 + alice_dh_4
x3dh_time_alice = time.time() - start_time
print(f"X3DH Time (Alice): {x3dh_time_alice:.6f} seconds")

X3DH Time (Alice): 0.001747 seconds


Bob computes his master secret

In [5]:
start_time = time.time()
# Bob also receives Alice's public keys when she starts the handshake
alice_bundle = alice.get_public_keys()

bob_dh_1 = bob.signed_pre_key.exchange(alice_bundle['ik'])
bob_dh_2 = bob.identity_key.exchange(alice_bundle['otks'][0])
bob_dh_3 = bob.signed_pre_key.exchange(alice_bundle['otks'][0])
bob_dh_4 = bob.ephemeral_keys[0].exchange(alice_bundle['otks'][0])

bob_ms = bob_dh_1 + bob_dh_2 + bob_dh_3 + bob_dh_4
x3dh_time_bob = time.time() - start_time
print(f"X3DH Time (Bob): {x3dh_time_bob:.6f} seconds")

X3DH Time (Bob): 0.000325 seconds


Derivating the master secret:

In [6]:
def key_derivation_function_root(master_secret):
    return HKDF(
        algorithm=hashes.SHA256(),
        length=64,
        salt=b'\x00' * 32,
        info=b"X3DH_algr",
    ).derive(master_secret)

In [7]:
alice_keys =  key_derivation_function_root(master_secret=alice_ms)
alice_root_key, alice_chain_key = alice_keys[:32], alice_keys[32:]

In [8]:
bob_keys = key_derivation_function_root(master_secret=bob_ms)
bob_root_key, bob_chain_key = bob_keys[:32], bob_keys[32:]

In [9]:
print(alice_root_key.hex())
print(bob_root_key.hex())

2614e64e137691304ac9051d6279d072946e7b7e58c32d294ab3fbb4c8867cac
2614e64e137691304ac9051d6279d072946e7b7e58c32d294ab3fbb4c8867cac


# Symmetric Ratcheting

In [10]:
def kdf_message(chain_key):
    msg_key = hmac.new(chain_key, b"\x01", hashlib.sha256).digest()
    next_chain_key = hmac.new(chain_key, b"\x02", hashlib.sha256).digest()
    return msg_key, next_chain_key

In [11]:
start_time = time.time()
alice_chain_key, msg_key_1 = kdf_message(alice_chain_key)
alice_chain_key, msg_key_2 = kdf_message(alice_chain_key)
sym_ratchet_time = time.time() - start_time

print(f"Symmetric Ratchet Time (2 steps): {sym_ratchet_time:.6f} seconds")
print(msg_key_1.hex())
print(msg_key_2.hex())

Symmetric Ratchet Time (2 steps): 0.000101 seconds
122c348092ea119910a5c73390619bacf9bf49c4f8871c161843b6c1ab4f2edd
ccb049fe2c6fe850725a21efd2dd8edf716b1c95f486bd2b71de86c509c23926


# Asymmetric Ratcheting

On initialization:

`root_key, chaining_key = kdf_root(master_secret, bob.shared_secret.exchange(alice.ratchet_key.public_key()))`

In [12]:
def kdf_root(root_key, shared_secret):
    result = hmac.new(root_key, shared_secret, hashlib.sha256).digest()
    return result[:16], result[16:]

In [13]:
start_time = time.time()
alice_ratchet = x25519.X25519PrivateKey.generate()
alice_root_key, alice_chain_key = kdf_root(alice_root_key, alice_ratchet.exchange(bob.identity_key.public_key()))
asym_ratchet_init_time = time.time() - start_time
print(f"Asymmetric Ratchet Init Time: {asym_ratchet_init_time:.6f} seconds")
print(alice_root_key.hex())


Asymmetric Ratchet Init Time: 0.000271 seconds
e6d927729003b928024d432bca4cbb43


From state initiator-receiver to receiver-initiator

```
temp, chaining_key = kdf_root(root_key, alice.ratchet_key_prev.exchange(bob.ratchet_key.public_key())
```

In [14]:
bob_ratchet_keys = x25519.X25519PrivateKey.generate()

temp, alice_chain_key_ri = kdf_root(alice_root_key, alice_ratchet.exchange(bob_ratchet_keys.public_key()))

print(alice_chain_key_ri.hex())

2fbbbd888001a750b082fb2eb6dfdd8a


From state receiver-initiator to initiator-receiver

```
root_key, chaining_key = kdf_root(temp, alice.ratchet_key.exchange(bob_ratchet_keys.public_key()))
```

In [15]:
alice_new_ratchet_keys = x25519.X25519PrivateKey.generate()

new_alice_root_key, alice_chain_key_ir = kdf_root(temp, alice_new_ratchet_keys.exchange(bob_ratchet_keys.public_key()))

print(new_alice_root_key.hex())

79ad9ccd22bfc348e6484690df4034d2


# Metric Analysis

In [16]:
# Key Sizes
identity_key_bytes = alice.identity_key.private_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PrivateFormat.Raw,
    encryption_algorithm=serialization.NoEncryption()
)
public_key_bytes = alice.identity_key.public_key().public_bytes(
    encoding=serialization.Encoding.Raw,
    format=serialization.PublicFormat.Raw
)

print(f"Private Key Size: {len(identity_key_bytes)} bytes")
print(f"Public Key Size: {len(public_key_bytes)} bytes")
print(f"Root Key Size: {len(alice_root_key)} bytes")
print(f"Chain Key Size: {len(alice_chain_key)} bytes")

Private Key Size: 32 bytes
Public Key Size: 32 bytes
Root Key Size: 16 bytes
Chain Key Size: 16 bytes


# Security Analysis


## Quantum Vulnerability

The current implementation of the Signal Protocol relies on the **X25519** Elliptic Curve Diffie-Hellman (ECDH) algorithm. 
While highly secure against classical attacks, it is vulnerable to **Shor's Algorithm** on a sufficiently powerful quantum computer. 

Shor's algorithm can solve the Elliptic Curve Discrete Logarithm Problem (ECDLP) in polynomial time, allowing an attacker to derive private keys from public keys.

### Attack Simulation (Conceptual)
1. **Eavesdropping:** An attacker captures Alice's public identity key and Bob's signed pre-key from the network.
2. **Quantum Computation:** The attacker uses a quantum computer to run Shor's algorithm on the public keys.
3. **Key Recovery:** The attacker retrieves the private keys and can now decrypt all past and future messages, breaking the Forward Secrecy and Post-Quantum Security properties.

To mitigate this, the protocol must be upgraded with **Lattice-based cryptography**, as explored in the Post-Quantum (PQ) notebooks of this project.